In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc

sys.path.append(os.path.abspath("../"))

from util import dataset as u_dataset

In [ ]:
log_path = Path("../../logs/fit/")

In [ ]:
def plot_pr_curve(run: int, mode: str, specification: str, object_name: str):
    # Your base directory
    base_dir = Path(log_path, mode, f"run_{run}", specification)

    # Get the single folder inside base_dir
    folders = [f for f in base_dir.iterdir() if f.is_dir()]
    if len(folders) == 1:
        path_to_model = folders[0]
    else:
        raise ValueError(f"Expected exactly one folder in {base_dir}, found {len(folders)}")

    # Load data
    recalls = np.load(path_to_model.as_posix() + f"/recalls/{object_name}.npy")
    precisions = np.load(path_to_model.as_posix() + f"/precisions/{object_name}.npy")
    title = f"Precision-Recall Curve, run: {run}, object: {object_name}"

    plot(recalls, precisions, title)


def plot(recalls, precisions, title):
    # # Compute AUC-PR
    pr_auc = auc(recalls, precisions)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(
        recalls,
        precisions,
        color="steelblue",
        linewidth=2,
        label=f"AP = {pr_auc:.3f})",
    )
    ax.fill_between(recalls, precisions, alpha=0.1, color="steelblue")

    # Baseline (random classifier)
    # ax.axhline(
    #     y=precisions.mean(),
    #     color="gray",
    #     linestyle="--",
    #     linewidth=1,
    #     label="Baseline (random)",
    # )

    ax.set_xlabel("Recall", fontsize=13)
    ax.set_ylabel("Precision", fontsize=13)
    ax.set_title(title, fontsize=15)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig("precision_recall_curve.pdf", dpi=150)
    plt.show()

In [ ]:
def plot_curves(run, mode, specification, categories):
    for object in categories:
        if object == u_dataset.CategoryNames.INTERSECTIONS:
            for type in list(u_dataset.IntersectionType)[1:]:
                object_name = f"{object.value}_{type.name}"
                plot_pr_curve(run, mode, specification, object_name)

        else:
            plot_pr_curve(run, mode, specification, object.value)

In [ ]:
categories = [
    u_dataset.CategoryNames.BALL,
    u_dataset.CategoryNames.PENALTYMARK,
    # u_dataset.CategoryNames.INTERSECTIONS,
]

mode = "classifier-basic"
specification = "v4"

for run in range(1, 2):
    plot_curves(run, mode, specification, categories)

In [ ]:

mode = "classifier-saturation"
specification = "32"

for run in range(1, 2):
    plot_curves(run, mode, specification, categories)

### Demonstration

In [ ]:
np.random.seed(42)

recalls = np.linspace(0.0, 1, 100)

precisions = np.zeros(100)

for i, r in enumerate(recalls):
    if r < 0.15:
        base = 0.50
        noise = np.random.uniform(-0.04, 0.04)
    elif r < 0.75:
        base = 0.50 + (r - 0.15) * (0.47 / 0.60)  # linear rise from 0.50 to ~0.97
        noise = np.random.uniform(-0.04, 0.04)
    elif r < 0.85:
        # small valley: dips down then recovers
        mid = 0.80
        depth = 0.12
        base = 1.0 - depth * np.exp(-((r - mid) ** 2) / (2 * 0.02))
        noise = np.random.uniform(-0.02, 0.02)
    else:
        base = 1.0 - (r - 0.85) * (0.70 / 0.15)   # sharp collapse
        noise = np.random.uniform(-0.02, 0.02)

    precisions[i] = np.clip(base + noise, 0.0, 1.0)

print("recalls =", recalls.tolist())
print("precisions =", precisions.tolist())

precisions_interp = np.maximum.accumulate(precisions[::-1])[::-1]

# plot(recalls, precisions, "")
plot(recalls, precisions_interp, "")